In [ ]:
# 필수 라이브러리 설치 (필요시 주석 해제 후 실행)
# !pip install fastf1 pandas numpy scikit-learn torch matplotlib

import fastf1
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
import matplotlib.pyplot as plt

# 1. 설정 (Configuration)
EVENT = 'Japanese Grand Prix'
YEAR = 2023
SESSION = 'R'
CACHE_DIR = '../fastf1_cache'  # 캐시 경로 설정
TARGET_DRIVERS = ['VER', 'NOR', 'PIA', 'LEC', 'HAM', 'SAI', 'RUS', 'ALO', 'OCO', 'GAS']
DIST_POINTS = 1000  # 한 랩당 고정할 거리 포인트 개수 (예: 200개 구간)

fastf1.Cache.enable_cache(CACHE_DIR)

# 2. 세션 로드
session = fastf1.get_session(YEAR, EVENT, SESSION)
session.load()

print(f"Session Loaded: {YEAR} {EVENT} - {SESSION}")

core           INFO 	Loading data for Japanese Grand Prix - Race [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 1 completed the race distance 00:00.076000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '81', '16', '44', '55', '63', '14', '31', '10', '40', '2

Session Loaded: 2023 Japanese Grand Prix - R


In [3]:
def calculate_slip_delta(telemetry_df, clean_laps_idx):
    """
    명세 4-B: SLIP_DELTA 계산
    각 기어별 Speed/RPM 비율(Baseline)을 구해 미끄러짐 정도를 산출
    """
    # Baseline 계산을 위해 깨끗한 랩 데이터만 사용
    baseline_data = telemetry_df[telemetry_df['nGear'] > 0] # 중립/후진 제외
    
    # 각 기어별 평균 Speed/RPM 비율 계산
    gear_ratios = {}
    for gear in baseline_data['nGear'].unique():
        gear_slice = baseline_data[baseline_data['nGear'] == gear]
        # 0으로 나누기 방지
        valid_rpm = gear_slice[gear_slice['RPM'] > 0]
        if len(valid_rpm) > 0:
            ratio = (valid_rpm['Speed'] / valid_rpm['RPM']).mean()
            gear_ratios[gear] = ratio
    
    # 전체 데이터에 대해 Estimated Speed 및 SLIP_DELTA 계산
    # 기어가 0이거나 데이터가 없는 경우 0으로 처리
    telemetry_df['Estimated_Speed'] = telemetry_df.apply(
        lambda row: row['RPM'] * gear_ratios.get(row['nGear'], 0), axis=1
    )
    
    telemetry_df['SLIP_DELTA'] = telemetry_df['Estimated_Speed'] - telemetry_df['Speed']
    return telemetry_df

def process_driver_laps(driver, session, scaler_fit=False, scaler=None):
    """
    드라이버별 데이터 처리 파이프라인
    """
    print(f"Processing {driver}...")
    laps = session.laps.pick_driver(driver)
    
    # --- Step 1: Filtering (명세 4. Step 1) ---
    # 1. Pit In/Out 제외, TrackStatus Normal(1)
    clean_laps = laps.pick_wo_box().pick_track_status('1', how='all')
    
    # 2. Lap 1 제외
    clean_laps = clean_laps[clean_laps['LapNumber'] > 1]
    
    # 3. Clean Air Filter (간소화된 구현: 앞뒤 차량 간격 확인)
    # 실제 앞뒤 차량 간격(Gap)은 계산이 복잡하므로 여기서는
    # 'FastF1이 제공하는 신뢰할 수 있는 랩(pick_accurate)'을 기준으로 
    # 섹터 타임이 안정적인 랩을 1차적으로 선정합니다.
    # *참고: 정확한 Gap > 1.5s 구현을 위해서는 전체 드라이버 위치 병합이 필요합니다.
    clean_laps = clean_laps.pick_accurate() 
    
    processed_laps = []
    
    for lap_idx, lap in clean_laps.iterrows():
        try:
            # 텔레메트리 로드 (거리 기반 보간 전)
            # car_data와 pos_data 병합
            telemetry = lap.get_telemetry()
            
            # --- Step 3: Resampling (Distance-Based) ---
            # 거리를 기준으로 데이터 보간 (예: 총 트랙 길이 5807m -> 200개 포인트)
            # make_new_length는 시간을 기준으로 하므로, 거리 기준으로 직접 보간해야 함
            
            # 1. 거리 축 생성 (0 ~ 1 사이로 정규화된 거리 or 미터 단위)
            total_dist = telemetry['Distance'].max()
            new_dist = np.linspace(0, total_dist, DIST_POINTS)
            
            # 2. 보간할 컬럼들
            cols_to_interp = ['Speed', 'RPM', 'Throttle', 'nGear', 'DRS']
            
            new_data = {'Distance': new_dist}
            for col in cols_to_interp:
                new_data[col] = np.interp(new_dist, telemetry['Distance'], telemetry[col])
            
            # Brake는 Boolean일 수 있으므로 0.5 이상을 1로 처리하거나 그대로 사용
            new_data['Brake'] = np.interp(new_dist, telemetry['Distance'], telemetry['Brake'].astype(float))
            
            resampled_df = pd.DataFrame(new_data)
            
            # --- Step 2: Feature Engineering ---
            # SLIP_DELTA 계산 (이 랩 내에서 계산)
            resampled_df = calculate_slip_delta(resampled_df, None)
            
            # Context Features 추가
            resampled_df['TyreLife'] = lap['TyreLife']
            resampled_df['LapNumber'] = lap['LapNumber']
            
            # Compound One-Hot Encoding (수동 처리)
            resampled_df['Compound_SOFT'] = 1 if lap['Compound'] == 'SOFT' else 0
            resampled_df['Compound_MEDIUM'] = 1 if lap['Compound'] == 'MEDIUM' else 0
            resampled_df['Compound_HARD'] = 1 if lap['Compound'] == 'HARD' else 0
            
            processed_laps.append(resampled_df)
            
        except Exception as e:
            continue

    if not processed_laps:
        return None

    # 모든 랩을 하나의 DataFrame으로 합침
    full_df = pd.concat(processed_laps, ignore_index=True)
    return full_df

# --- 전체 데이터셋 구축 ---
all_data_list = []

for driver in TARGET_DRIVERS:
    driver_df = process_driver_laps(driver, session)
    if driver_df is not None:
        all_data_list.append(driver_df)

full_dataset = pd.concat(all_data_list, ignore_index=True)

# --- Scaling (MinMax) ---
# 명세 5: Final Feature Set Columns
feature_cols = [
    'Speed', 'RPM',       # Physics
    'Throttle', 'Brake', 'nGear', 'DRS', # Control
    'SLIP_DELTA',         # Engineered
    'TyreLife', 'LapNumber', 'Compound_SOFT', 'Compound_MEDIUM', 'Compound_HARD' # Context
]

scaler = MinMaxScaler()
full_dataset[feature_cols] = scaler.fit_transform(full_dataset[feature_cols])

print(f"Data Processing Complete. Shape: {full_dataset.shape}")

Processing VER...


c:\Users\GC\Documents\3-2win\1. 시계열\시계열 팀플\f1_project\.venv\Lib\site-packages\fastf1\core.py:3183: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"


ValueError: Invalid value 'all' for kwarg 'how'

In [ ]:
# 1. 데이터 Reshape (Samples, Sequence_Length, Features)
# DIST_POINTS = Sequence Length
num_features = len(feature_cols)
num_samples = len(full_dataset) // DIST_POINTS

# 데이터를 [N, 200, Features] 형태로 변환
X_data = full_dataset[feature_cols].values
X_reshaped = X_data[:num_samples * DIST_POINTS].reshape(num_samples, DIST_POINTS, num_features)

# PyTorch Tensor 변환
# Autoencoder는 입력이 곧 타겟이므로 X와 y가 동일
tensor_X = torch.FloatTensor(X_reshaped)

# DataLoader 생성
BATCH_SIZE = 32
dataset = TensorDataset(tensor_X, tensor_X)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# 2. LSTM Autoencoder 모델 정의
class LSTMAutoencoder(nn.Module):
    def __init__(self, seq_len, n_features, embedding_dim=64):
        super(LSTMAutoencoder, self).__init__()
        
        self.seq_len = seq_len
        self.n_features = n_features
        self.embedding_dim = embedding_dim
        
        # Encoder
        self.encoder_lstm = nn.LSTM(
            input_size=n_features, 
            hidden_size=embedding_dim, 
            num_layers=1, 
            batch_first=True
        )
        
        # Decoder
        self.decoder_lstm = nn.LSTM(
            input_size=embedding_dim, 
            hidden_size=embedding_dim, 
            num_layers=1, 
            batch_first=True
        )
        
        self.output_layer = nn.Linear(embedding_dim, n_features)

    def forward(self, x):
        # Encoder: 입력 시퀀스를 압축하여 Hidden state 생성
        # output: (batch, seq, hidden), (h_n, c_n)
        _, (hidden, _) = self.encoder_lstm(x)
        
        # Decoder 입력 준비: Encoder의 마지막 Hidden state를 반복하여 시퀀스로 만듦
        # hidden[-1] shape: (batch, embedding_dim)
        # repeat -> (batch, seq_len, embedding_dim)
        x_repeated = hidden[-1].unsqueeze(1).repeat(1, self.seq_len, 1)
        
        # Decoder 실행
        decoder_output, _ = self.decoder_lstm(x_repeated)
        
        # 원래 Feature 차원으로 복원
        x_reconstructed = self.output_layer(decoder_output)
        
        return x_reconstructed

# 모델 초기화
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LSTMAutoencoder(seq_len=DIST_POINTS, n_features=num_features, embedding_dim=128).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(f"Model Structure: \n{model}")
print(f"Training on device: {device}")

In [ ]:
NUM_EPOCHS = 50
loss_history = []

print("Start Training...")
model.train()

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0
    for batch_x, _ in dataloader:
        batch_x = batch_x.to(device)
        
        # Forward
        output = model(batch_x)
        loss = criterion(output, batch_x)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(dataloader)
    loss_history.append(avg_loss)
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}], Loss: {avg_loss:.6f}")

# 학습 결과 시각화
plt.figure(figsize=(10, 5))
plt.plot(loss_history, label='Training Loss')
plt.title('LSTM Autoencoder Training Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.show()